# Анализ эксперимента по подстройке коэффициентов Kp и Ki от 28 ноября 2025


Используется модель, обученная 25 ноября 2025 г.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nn_laser_stabilizer.config.config import load_config
from nn_laser_stabilizer.utils.paths import get_experiment_dir

EXPERIMENT_NAME = "pid_delta_tuning-inference"
EXPERIMENT_DATE = "2025-11-28"
EXPERIMENT_TIME = "16-15-57"

EXPERIMENT_DIR = get_experiment_dir(
    experiment_name=EXPERIMENT_NAME,
    experiment_date=EXPERIMENT_DATE,
    experiment_time=EXPERIMENT_TIME,
)
CONFIG_PATH = EXPERIMENT_DIR / "config.yaml"
ENV_LOG_PATH = EXPERIMENT_DIR / "env_logs" / "env.log"
TRAIN_LOG_PATH = EXPERIMENT_DIR / "logs" / "train.log"
CONNECTION_LOG_PATH = EXPERIMENT_DIR / "connection_logs" / "connection.log"

config = load_config(CONFIG_PATH)
print(f"Эксперимент: {config.experiment_name}")
print(f"Seed: {config.seed}")

## Анализ логов окружения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_keyval_log

# env.log содержит строки двух типов: шаги (есть should_reset) и reset (нет).
# parse_keyval_log читает все строки; ниже воспроизводим прежнюю логику:
#   - классификация step/reset по наличию should_reset;
#   - reset-строки наследуют номер последнего шага (как current_step).
_raw = parse_keyval_log(ENV_LOG_PATH)
_raw = _raw[
    _raw[["kp", "ki", "kd", "error_mean_norm", "error_std_norm"]].notna().all(axis=1)
].reset_index(drop=True)

is_step = _raw["should_reset"].notna()

env_df = pd.DataFrame({
    "Step": _raw["step"],
    "time": _raw["time"],
    "Type": np.where(is_step, "step", "reset"),
    "Kp": _raw["kp"],
    "Ki": _raw["ki"],
    "Kd": _raw["kd"],
    "Delta Kp": _raw["delta_kp_norm"].where(is_step),
    "Delta Ki": _raw["delta_ki_norm"].where(is_step),
    "Error mean norm": _raw["error_mean_norm"],
    "Error std norm": _raw["error_std_norm"],
    "Reward": _raw["reward"].where(is_step),
    "Should reset": _raw["should_reset"].where(is_step, other=True).astype(bool),
})
env_df["Step"] = env_df["Step"].ffill().fillna(0).astype(int)
reset_steps = env_df.loc[env_df["Type"] == "reset", "Step"].tolist()

print(f"Загружено {len(env_df)} записей из логов окружения")
print(f"Найдено {len(reset_steps)} reset событий")
print(f"Диапазон шагов: {env_df['Step'].min()} - {env_df['Step'].max()}")

In [ ]:
step_df = env_df[env_df['Type'] == 'step'].copy()
print("=== Статистика по шагам окружения ===")
print(step_df.describe())


In [ ]:
columns_to_plot = ['Kp', 'Ki', 'Error mean norm', 'Error std norm', 'Reward']

for col in columns_to_plot:
    plt.figure(figsize=(12, 5))
    plt.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=2, label=col)
    
    for reset_step in reset_steps:
        if reset_step <= step_df['Step'].max():
            plt.axvline(x=reset_step, color='red', linestyle=':', linewidth=2, alpha=0.6)
    
    plt.title(f'{col} over Steps')
    plt.xlabel('Step')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=step_df['Kp'], y=step_df['Error std norm'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error Std Norm')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=step_df['Kp'], y=step_df['Error mean norm'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error mean norm')
cur_ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=step_df['Ki'], y=step_df['Error std norm'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error Std Norm')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=step_df['Ki'], y=step_df['Error mean norm'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error mean norm')
cur_ax.grid(True, alpha=0.3)


In [ ]:
corr_matrix = step_df[['Kp', 'Ki', 'Error mean norm', 'Error std norm', 'Reward']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


## Анализ состояния установки


In [ ]:
block_size = config.env.args.block_size
for i in range(1, len(reset_steps)):
    reset_steps[i] = reset_steps[i] * block_size + 1000  

In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_connection_log

# parse_connection_log отдаёт сырые записи SEND/READ (тип — в колонке 'event').
# Историческое поведение: i-я посылка (SEND) спаривается с i-м чтением (READ).
conn_raw = parse_connection_log(CONNECTION_LOG_PATH)

send_df = conn_raw[conn_raw["event"] == "SEND"][
    ["kp", "ki", "kd", "control_min", "control_max"]
].reset_index(drop=True)
send_df["step"] = range(len(send_df))

read_df = conn_raw[conn_raw["event"] == "READ"][
    ["process_variable", "control_output"]
].reset_index(drop=True)
read_df["step"] = range(len(read_df))

connection_df = send_df.merge(read_df, on="step", how="left")
print(f"Загружено {len(connection_df)} записей из логов соединения")
if len(connection_df) > 0:
    print(f"Диапазон шагов: {connection_df['step'].min()} - {connection_df['step'].max()}")

In [ ]:
if len(connection_df) > 0:
    setpoint = config.env.args.setpoint
    
    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')
    plt.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint ({setpoint})')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='red', linestyle=':', linewidth=2, alpha=0.6)
    
    plt.title('Process Variable')
    plt.xlabel('Step')
    plt.ylim(500, 1700)
    plt.ylabel('Process Variable')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='red', linestyle=':', linewidth=2, alpha=0.6)
    
    plt.title('Control Output')
    plt.xlabel('Step')
    plt.ylabel('Control Output')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['kp'], 'r-', alpha=0.7, linewidth=0.8, label='Kp')
    plt.plot(connection_df['step'], connection_df['ki'], 'g-', alpha=0.7, linewidth=0.8, label='Ki')
    plt.plot(connection_df['step'], connection_df['kd'], 'b-', alpha=0.7, linewidth=0.8, label='Kd')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='red', linestyle=':', linewidth=2, alpha=0.6)
    
    plt.title('PID Coefficients')
    plt.xlabel('Step')
    plt.ylabel('Coefficient Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if len(connection_df) > 0:
    corr_matrix = connection_df[['kp', 'ki', 'control_output', 'process_variable']].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix (Connection Data)')
    plt.tight_layout()
    plt.show()


In [ ]:
env_df = env_df.sort_values('time').copy()

env_df['time_diff'] = env_df['time'].diff()
env_df['time_diff_ms'] = env_df['time_diff'] * 1000  


step_time_df = env_df[env_df['Type'] == 'step'].copy()
time_df = env_df.copy()

print("=== Статистика по времени ===")
print(f"Всего записей: {len(time_df)}")
print(f"Шагов: {len(time_df[time_df['Type'] == 'step'])}")
print(f"Reset событий: {len(time_df[time_df['Type'] == 'reset'])}")

if len(step_time_df) > 0:
    print(f"\n=== Статистика интервалов между шагами ===")
    print(step_time_df['time_diff_ms'].describe())
    print(f"\nМедианный интервал: {step_time_df['time_diff_ms'].median():.2f} мс")
    print(f"Средний интервал: {step_time_df['time_diff_ms'].mean():.2f} мс")
    print(f"Максимальный интервал: {step_time_df['time_diff_ms'].max():.2f} мс")
    print(f"Минимальный интервал: {step_time_df['time_diff_ms'].min():.2f} мс")
    
    plt.figure(figsize=(12, 5))
    plt.plot(step_time_df['Step'], step_time_df['time_diff_ms'], 'b-', alpha=0.7, linewidth=0.8)
    plt.title('Time Intervals Between Steps')
    plt.xlabel('Step')
    plt.ylabel('Time Interval (ms)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 5))
    plt.hist(step_time_df['time_diff_ms'].dropna(), bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Time Intervals Between Steps')
    plt.xlabel('Time Interval (ms)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
reset_time_df = env_df[env_df['Type'] == 'reset'].copy()
if len(reset_time_df) > 0:
    print(f"\n=== Статистика интервалов между шагами ===")
    print(reset_time_df['time_diff_ms'].describe())
    print(f"\nМедианный интервал: {reset_time_df['time_diff_ms'].median():.2f} мс")
    print(f"Средний интервал: {reset_time_df['time_diff_ms'].mean():.2f} мс")
    print(f"Максимальный интервал: {reset_time_df['time_diff_ms'].max():.2f} мс")
    print(f"Минимальный интервал: {reset_time_df['time_diff_ms'].min():.2f} мс")
    
    plt.figure(figsize=(12, 5))
    plt.plot(reset_time_df['Step'], reset_time_df['time_diff_ms'], 'b-', alpha=0.7, linewidth=0.8)
    plt.title('Time Intervals Between Steps')
    plt.xlabel('Step')
    plt.ylabel('Time Interval (ms)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 5))
    plt.hist(reset_time_df['time_diff_ms'].dropna(), bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Time Intervals Between Steps')
    plt.xlabel('Time Interval (ms)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### График для встречи

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({'font.size': 14})  

columns_left = ['Kp', 'Ki']
colors_left = ['green', 'blue']  
column_right = 'Error std norm'

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(2, 2, width_ratios=[1, 1.2], height_ratios=[1,1], wspace=0.3, hspace=0.4)

for i, col in enumerate(columns_left):
    ax = fig.add_subplot(gs[i,0])
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])
    
    ax.set_ylabel(col)
    ax.set_xlabel('Block')
    ax.grid(True, alpha=0.3)

ax_right = fig.add_subplot(gs[:,1])  
ax_right.plot(step_df['Step'], step_df[column_right], alpha=0.8, linewidth=1.5, color='purple', label=column_right)


ax_right.set_xlabel('Block')
ax_right.set_ylabel(column_right)
ax_right.grid(True, alpha=0.3)
ax_right.legend()

fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.4)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({'font.size': 14})

setpoint = 1200

columns_left = ['process_variable']
columns_right = ['control_output']
fig = plt.figure(figsize=(18, 6))
gs = GridSpec(1, 2, width_ratios=[1, 1], wspace=0.3)

ax_left = fig.add_subplot(gs[0,0])
ax_left.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=1.2, label='Process Variable')
ax_left.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint')
ax_left.set_xlim(left=100)
ax_left.set_ylim(0, 1800)

ax_left.set_xlabel('Step')
ax_left.set_ylabel('Process Variable')
ax_left.set_title('Process Variable')
ax_left.grid(True, alpha=0.3)
ax_left.legend()

ax_right = fig.add_subplot(gs[0,1])
ax_right.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=1.2, label='Control Output')
ax_right.set_xlim(left=100)

ax_right.set_xlabel('Step')
ax_right.set_ylabel('Control Output')
ax_right.set_title('Control Output')
ax_right.grid(True, alpha=0.3)
ax_right.legend()

plt.tight_layout()
plt.show()
